## 1 读取负荷数据和光伏数据

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings("ignore")

#设置随机种子
def set_random_seed(seed_value):
    """设置随机种子"""
    np.random.seed(seed_value)  # NumPy
    random.seed(seed_value)  # Python
    torch.manual_seed(seed_value)  # PyTorch CPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)  # PyTorch GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

#设置随机种子，确保结果可复现⭐⭐⭐
set_random_seed(42)
#设置设备
device='cuda' if torch.cuda.is_available() else 'cpu'

# --------------------------
# 读取数据
# --------------------------
data_file = "Annual_load_PV_15min.csv"  # CSV 文件路径
df = pd.read_csv(data_file, parse_dates=["Time"])
df.set_index("Time", inplace=True)

pv_profile = df["PV_kW"].values.astype(np.float32)    # 光伏功率序列
load_profile = df["Load_kW"].values.astype(np.float32) # 负载功率序列

# 爱沙尼亚峰谷电价(€/kWh)
price_profile = np.where((df.index.hour >= 17) & (df.index.hour <= 20),
                         0.30,  # 高峰
                         0.20).astype(np.float32)  # 谷时

# --------------------------
# 找到光伏出力最大值所在的日期
# --------------------------
df['PV_kW_float'] = pv_profile  # 临时列用于求每日最大值
daily_pv_max = df['PV_kW_float'].resample('D').max()  # 每天最大光伏功率
max_pv_day = daily_pv_max.idxmax().date()            # 光伏最大值对应日期

# 前一天日期
from datetime import timedelta
prev_day = max_pv_day - timedelta(days=1)

print(f"光伏出力最大值日期: {max_pv_day}, 前一天日期: {prev_day}")

# --------------------------
# 按日期划分训练集和测试集（两天）
# --------------------------
test_mask = (df.index.date == max_pv_day) | (df.index.date == prev_day)

pv_test = pv_profile[test_mask]
load_test = load_profile[test_mask]
price_test = price_profile[test_mask]

pv_train = pv_profile[~test_mask]
load_train = load_profile[~test_mask]
price_train = price_profile[~test_mask]

print(f"训练集长度: {len(pv_train)}, 测试集长度: {len(pv_test)}")

## 2 环境定义

In [ ]:
# 定义储能环境类，继承自 gym.Env
class EnergyStorageEnv(gym.Env):
    def __init__(self, pv, load, price, e_cap=10, p_max=2, dt=0.25,
                 eta_ch=1, eta_dis=1):
        super().__init__()
        self.pv = pv        # 光伏发电功率序列
        self.load = load    # 负荷功率序列
        self.price = price  # 电价序列
        self.T = len(pv)    # 总时间步数
        self.dt = dt        # 每步时长（小时）
        self.e_cap = e_cap  # 储能容量（kWh）
        self.p_max = p_max  # 最大充放电功率（kW）
        self.soc_min = 0.1  # 最小SOC
        self.soc_max = 0.9  # 最大SOC
        self.eta_ch = eta_ch
        self.eta_dis = eta_dis

        # 观测空间: SOC, 负荷, 光伏
        self.observation_space = spaces.Box(
            low=np.array([0., 0., 0.], dtype=np.float32),
            high=np.array([1., np.max(load), np.max(pv)], dtype=np.float32),
            dtype=np.float32
        )

        # 动作空间: 0放电, 1保持, 2充电
        self.action_space = spaces.Discrete(3)

    def reset(self):
        self.t = 0
        self.soc = 0.5
        self.soc_history = []
        self.grid_history = []
        self.pv_history = []
        self.load_history = []

        obs = np.array([self.soc, self.load[self.t], self.pv[self.t]], dtype=np.float32)
        return obs, {}

    def step(self, action):
        # 动作映射
        p_map = {0: -self.p_max, 1: 0.0, 2: self.p_max}
        p_batt = p_map[action]

        # 修正电池功率使其不超过SOC边界
        if p_batt > 0:  # 充电
            p_batt = min(p_batt, (self.soc_max - self.soc) * self.e_cap / self.dt / self.eta_ch)
        elif p_batt < 0:  # 放电
            p_batt = max(p_batt, (self.soc_min - self.soc) * self.e_cap / self.dt * self.eta_dis)

        # 更新SOC，考虑效率
        if p_batt >= 0:
            self.soc += p_batt * self.dt / self.e_cap * self.eta_ch
        else:
            self.soc += p_batt * self.dt / self.e_cap / self.eta_dis
        self.soc = np.clip(self.soc, self.soc_min, self.soc_max)

        # 计算网购电（假设不能卖电）
        net_load = self.load[self.t] - self.pv[self.t] - p_batt
        grid_import = max(net_load, 0)

        # 电网购电成本
        grid_cost = grid_import * self.price[self.t] * self.dt
        # 奖励函数：最大化电网购电成本的相反数
        reward = -grid_cost

        # 记录历史
        self.soc_history.append(self.soc)
        self.grid_history.append(grid_import)
        self.pv_history.append(self.pv[self.t])
        self.load_history.append(self.load[self.t])

        # 时间步推进
        self.t += 1
        terminated = self.t >= self.T
        truncated = False

        # 下一个观测值
        if not terminated:
            obs = np.array([self.soc, self.load[self.t], self.pv[self.t]], dtype=np.float32)
        else:
            obs = np.array([self.soc, 0, 0], dtype=np.float32)

        info = {
            "pv": self.pv[self.t - 1] if self.t - 1 < self.T else 0,
            "load": self.load[self.t - 1] if self.t - 1 < self.T else 0
        }

        return obs, reward, terminated, truncated, info

#创建训练和测试环境
env_train = EnergyStorageEnv(pv_train, load_train, price_train)
env_test  = EnergyStorageEnv(pv_test, load_test, price_test)

## 可视化的相关函数（四种策略均适用）

In [ ]:
# 训练过程中累计奖励的实时可视化
def plot_reward_live(rewards, title="Live Reward"):
    plt.figure(figsize=(8,4))
    plt.plot(rewards, marker='o', color='blue')
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.grid(True)
    plt.show()

def plot_reward_animation(rewards, title="Reward Animation", save_path=None, fps=5, max_frames=100):
    """
    绘制奖励变化的动态动画
    
    参数:
        rewards (list or np.array): 奖励序列
        title (str): 图表标题，默认 "Reward Animation"
        save_path (str or None): 如果提供路径，则保存为视频文件
        fps (int): 保存视频的帧率
        max_frames (int): 动画最多显示的帧数（如果奖励序列太长，会抽样显示）
    """
    total_steps = len(rewards)  # 总的时间步数 / 奖励点数

    # -------------------------------
    # 抽样帧（如果奖励序列太长，避免动画过慢）
    if total_steps > max_frames:
        stride = int(np.ceil(total_steps / max_frames))  # 计算采样步长
        frame_indices = list(range(0, total_steps, stride))  # 生成要显示的帧索引列表
        if frame_indices[-1] != total_steps - 1:
            # 确保最后一步奖励也被显示
            frame_indices.append(total_steps - 1)
    else:
        # 奖励序列不长，全部显示
        frame_indices = list(range(total_steps))

    # -------------------------------
    # 创建绘图对象
    fig, ax = plt.subplots(figsize=(10, 4))  # 图像大小 10x4 英寸
    line, = ax.plot([], [], color='blue', marker='o', label='Reward')  # 初始化折线对象，初始为空

    # 设置坐标轴范围
    ax.set_xlim(0, total_steps)  # x轴范围: 0 ~ 总步数
    ax.set_ylim(min(rewards) * 1.1, max(rewards) * 0.9)  # 因为此处的reward是负数，因此最大值*0.9，最小值*1.1
    ax.set_xlabel("Episode / Time Step")  # x轴标签
    ax.set_ylabel("Reward")               # y轴标签
    ax.grid(True)                         # 显示网格
    ax.set_title(title)                   # 设置标题
    ax.legend()                           # 显示图例

    # -------------------------------
    # 动画初始化函数
    def init():
        line.set_data([], [])  # 初始折线为空
        return (line,)

    # -------------------------------
    # 动画每帧更新函数
    def update(i):
        idx = frame_indices[i]           # 当前帧对应奖励索引
        x = range(idx + 1)               # x 轴范围：从 0 到当前帧
        line.set_data(x, rewards[:idx + 1])  # 更新折线数据
        return (line,)

    # -------------------------------
    # 创建动画对象
    ani = FuncAnimation(fig, update, frames=len(frame_indices),
                        init_func=init, interval=100, blit=False)  # interval=100ms每帧间隔

    # 在 Jupyter Notebook 中显示动画
    from IPython.display import HTML, display
    display(HTML(ani.to_jshtml()))
    plt.close(fig)  # 关闭静态图，防止重复显示

    # -------------------------------
    # 可选：保存动画为视频文件（如mp4）
    if save_path:
        ani.save(save_path, writer="ffmpeg", fps=fps)


# --------------------------
# 函数2：绘制SOC和Grid Power动画
# --------------------------
def plot_soc_power_animation(soc_history, grid_history, load_history, pv_history,
                             title="SOC & Power Animation", save_path=None, fps=5, max_frames=100):
    total_steps = len(soc_history)
    assert len(soc_history) == len(grid_history) == len(load_history) == len(pv_history), \
        "SOC 和各功率长度必须一致"

    # 抽样帧
    if total_steps > max_frames:
        stride = int(np.ceil(total_steps / max_frames))
        frame_indices = list(range(0, total_steps, stride))
        if frame_indices[-1] != total_steps - 1:
            frame_indices.append(total_steps - 1)
    else:
        frame_indices = list(range(total_steps))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # SOC 曲线
    line_soc, = axes[0].plot([], [], color='green', marker='o', label='SOC',markersize=3)

    # 多功率曲线
    line_grid, = axes[1].plot([], [], color='red', marker='o', label='Grid Power',markersize=2)
    line_load, = axes[1].plot([], [], color='blue', marker='o', label='Load Power',markersize=2)
    line_pv,   = axes[1].plot([], [], color='orange', marker='o', label='PV Power',markersize=2)

    # 设置坐标轴范围
    axes[0].set_xlim(0, total_steps)
    axes[0].set_ylim(min(soc_history)*0.9, max(soc_history)*1.1)
    axes[0].set_ylabel("SOC")
    axes[0].set_title("Battery SOC")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].set_xlim(0, total_steps)
    all_power = grid_history + load_history + pv_history
    axes[1].set_ylim(min(all_power)*0.9, max(all_power)*1.1)
    axes[1].set_xlabel("Episode / Time Step")
    axes[1].set_ylabel("Power (kW)")
    axes[1].set_title("Grid, Load, and PV Power")
    axes[1].grid(True)
    axes[1].legend()

    fig.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 初始化
    def init():
        line_soc.set_data([], [])
        line_grid.set_data([], [])
        line_load.set_data([], [])
        line_pv.set_data([], [])
        return line_soc, line_grid, line_load, line_pv

    # 更新每一帧
    def update(i):
        idx = frame_indices[i]
        x = range(idx + 1)
        line_soc.set_data(x, soc_history[:idx + 1])
        line_grid.set_data(x, grid_history[:idx + 1])
        line_load.set_data(x, load_history[:idx + 1])
        line_pv.set_data(x, pv_history[:idx + 1])
        return line_soc, line_grid, line_load, line_pv

    ani = FuncAnimation(fig, update, frames=len(frame_indices),
                        init_func=init, interval=50, blit=False)

    display(HTML(ani.to_jshtml()))
    plt.close(fig)

    if save_path:
        ani.save(save_path, writer="ffmpeg", fps=fps)

## 3 Q-Learning 算法

### 3.1 Q-Learning构建与训练

In [ ]:
# -------------------------------
# 状态离散化（Q-learning用）
# -------------------------------
# 将连续的SOC、负载和光伏出力离散化为有限的状态格点，用于Q表索引
soc_bins = np.linspace(env_train.soc_min, env_train.soc_max, 8)   # 将SOC分为8个区间
load_bins = np.linspace(0, max(load_train), 12)                    # 将负载分为12个区间
pv_bins = np.linspace(0, max(pv_train), 12)                        # 将PV出力分为12个区间

# 定义离散化函数：将连续状态映射到离散索引
def discretize_state(s):
    return (np.digitize(s[0], soc_bins)-1,   # SOC索引，digitize返回的是第几个bin，减1使索引从0开始
            np.digitize(s[1], load_bins)-1,  # 负载索引
            np.digitize(s[2], pv_bins)-1)    # PV出力索引

# -------------------------------
# 训练总轮数和强化学习参数
# -------------------------------
total_episodes = 200   # Q-learning训练总轮数
lr = 0.1              # 学习率
gamma = 0.90          # 折扣因子（未来奖励折扣）
eps = 0.1             # epsilon-greedy探索率
batch_size = 32       # batch大小（此处未用，预留）

# -------------------------------
# 初始化Q-learning表
# -------------------------------
q_table = np.zeros((8,12,12,3))   # Q表，状态维度为(8,12,12)，动作维度为3
q_rewards, q_soc_history, q_grid_history = [], [], []  # 用于记录训练奖励、SOC和电网功率历史

# -------------------------------
# Q-learning训练循环
# -------------------------------
for ep in range(total_episodes):
    s, _ = env_train.reset()                      # 重置环境，获取初始状态
    idx = discretize_state(s)                     # 离散化初始状态
    done = False                                  # episode完成标志
    total_reward = 0                              # 累计奖励初始化
    env_train.soc_history, env_train.grid_history = [], []   # 清空历史记录

    while not done:
        # epsilon-greedy策略选择动作
        if random.random() < eps:
            a = env_train.action_space.sample()  # 随机选择动作（探索）
        else:
            a = np.argmax(q_table[idx])          # 选择Q值最大的动作（利用）
        
        # 与环境交互，执行动作
        ns, r, terminated, truncated, _ = env_train.step(a)  # ns:下一个状态, r:奖励, terminated/truncated:结束标志
        done = terminated or truncated           # 判断是否结束
        nidx = discretize_state(ns)             # 将下一个状态离散化
        
        # Q-learning核心更新公式
        q_table[idx][a] += lr * (r + gamma*np.max(q_table[nidx]) - q_table[idx][a])
        
        idx = nidx                              # 状态更新
        total_reward += r                        # 累加奖励

    # 每轮训练结束后记录奖励
    q_rewards.append(total_reward)
    
    # 动态显示训练奖励曲线
    clear_output(wait=True)
    plot_reward_live(q_rewards, title="Q-Learning Live Reward")

# 绘制训练结束的完整动画曲线
plot_reward_animation(q_rewards,"Q-Learning Training")

### 3.2 Q-Learning测试

In [ ]:
# -------------------------------
# Q-learning测试阶段
# -------------------------------
q_test_rewards = []
q_total_reward=0                            # 用于记录测试奖励
s, _ = env_test.reset()                        # 重置测试环境
done = False
idx = discretize_state(s)                      # 离散化初始状态

while not done:
    a = np.argmax(q_table[idx])                # 测试阶段直接选择Q值最大的动作（贪心策略）
    ns, r, terminated, truncated, _ = env_test.step(a)
    done = terminated or truncated
    idx = discretize_state(ns)                 # 更新状态索引
    q_test_rewards.append(r)                   #保存奖励
    q_total_reward+=r

q_test_soc_history=env_test.soc_history.copy()
q_test_grid_history=env_test.grid_history.copy()
q_test_pv_history=env_test.pv_history.copy()
q_test_load_history=env_test.load_history.copy()

print(f"Q-Learning test total rewards: {q_total_reward:.2f}")

# 绘制测试阶段SOC、电网功率、光伏功率、负荷功率和奖励动画
plot_reward_animation(q_test_rewards,"Q-Learning Test")
plot_soc_power_animation(q_test_soc_history, q_test_grid_history, q_test_load_history, q_test_pv_history, title="Q-Learning Test")


## 4 DQN算法

### 4.1 DQN构建与训练

In [ ]:
# -------------------------------
# DQN神经网络定义
# -------------------------------
class DQNNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 定义一个简单的全连接网络，输入状态维度3（SOC, load, PV）
        # 输出动作值维度3（动作空间3个动作）
        self.fc = nn.Sequential(
            nn.Linear(3,64),  # 输入3维，隐藏层64个神经元
            nn.ReLU(),         # 激活函数ReLU
            nn.Linear(64,3)    # 输出3维Q值，对应3个动作
        )
    def forward(self,x):
        return self.fc(x.float())  # 前向传播，保证输入为float32

# -------------------------------
# 初始化网络、目标网络和优化器
# -------------------------------
dqn_net = DQNNet()                            # 在线网络（policy network）
target_net = DQNNet()                         # 目标网络（target network）
target_net.load_state_dict(dqn_net.state_dict())  # 初始化目标网络参数与在线网络一致
optimizer = optim.Adam(dqn_net.parameters(), lr=1e-2)  # Adam优化器
memory = deque(maxlen=5000)                   # 经验回放池，最多存5000条记录

# 初始化记录列表
dqn_rewards, dqn_soc_history, dqn_grid_history = [], [], []

# -------------------------------
# DQN训练循环
# -------------------------------
for ep in range(total_episodes):
    s, _ = env_train.reset()                   # 重置训练环境，获取初始状态
    done = False                               # 回合结束标志
    total_reward = 0                           # 累计奖励
    env_train.soc_history, env_train.grid_history = [], []  # 清空环境历史记录

    while not done:
        # epsilon-greedy策略选择动作
        if random.random() < eps:
            a = env_train.action_space.sample()  # 随机动作（探索）
        else:
            # 使用在线网络选择Q值最大的动作（利用）
            a = torch.argmax(dqn_net(torch.tensor(s,dtype=torch.float32))).item()

        # 与环境交互
        ns, r, terminated, truncated, _ = env_train.step(a)  # 执行动作，获得下一个状态和奖励
        done = terminated or truncated                          # 判断是否结束

        # 将下一个状态转为tensor，方便存入经验回放池
        ns_tensor = torch.tensor(ns,dtype=torch.float32)

        # 将当前经验存入经验回放池：状态、动作、奖励、下一状态、done
        memory.append((torch.tensor(s,dtype=torch.float32), a, r, ns_tensor, done))

        s = ns                       # 更新当前状态
        total_reward += r             # 累加奖励

        # 当经验回放池中数据足够一个batch时进行训练
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)      # 随机采样batch
            states, actions, rewards_batch, next_states, dones = zip(*batch)  # 解压
            states = torch.stack(states).float()          # 拼成tensor
            next_states = torch.stack(next_states).float()
            actions = torch.tensor(actions, dtype=torch.long)
            rewards_batch = torch.tensor(rewards_batch, dtype=torch.float32)
            dones = torch.tensor(dones, dtype=torch.float32)

            # 获取当前Q值对应的动作值
            q_values = dqn_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

            # 计算目标Q值，使用目标网络
            with torch.no_grad():
                q_next = target_net(next_states).max(1)[0]   # 下一状态最大Q值
            target = rewards_batch + gamma * q_next * (1 - dones)  # DQN目标

            # 计算损失
            loss = nn.MSELoss()(q_values, target)
            optimizer.zero_grad()   # 梯度清零
            loss.backward()         # 反向传播
            optimizer.step()        # 更新网络参数

    # 每轮训练结束后记录奖励
    dqn_rewards.append(total_reward)

    # 动态显示训练奖励曲线
    clear_output(wait=True)
    plot_reward_live(dqn_rewards, title="DQN Live Reward")

# 绘制训练完成的完整动画曲线
plot_reward_animation(dqn_rewards,"DQN Training")

### 4.2 DQN测试

In [ ]:
# -------------------------------
# DQN测试阶段
# -------------------------------
s, _ = env_test.reset()        # 重置测试环境
done = False
dqn_test_rewards=[]
dqn_total_reward=0

while not done:
    with torch.no_grad():
        # 测试阶段直接选择Q值最大的动作
        a = torch.argmax(dqn_net(torch.tensor(s,dtype=torch.float32))).item()
        ns, r, terminated, truncated, _ = env_test.step(a)
        done = terminated or truncated
        s = ns
        dqn_test_rewards.append(r)   # 保存奖励
        dqn_total_reward+=r         # 累加每步的奖励

dqn_test_soc_history=env_test.soc_history.copy()
dqn_test_grid_history=env_test.grid_history.copy()
dqn_test_pv_history=env_test.pv_history.copy()
dqn_test_load_history=env_test.load_history.copy()

print(f"DQN test total rewards: {dqn_total_reward:.2f}")

# 绘制测试阶段SOC、电网功率、光伏功率、负荷功率和奖励动画
plot_reward_animation(dqn_test_rewards,"DQN Test")
plot_soc_power_animation(dqn_test_soc_history, dqn_test_grid_history, dqn_test_load_history, dqn_test_pv_history, title="DQN Test")


## 5 Policy Gradient

### 5.1 PG模型构建与训练

In [ ]:
# -------------------------------
# 策略网络定义（Policy Gradient）
# -------------------------------
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 定义全连接网络：输入3维状态 -> 64隐藏 -> 3输出动作概率
        # 输出经过softmax转换为概率分布
        self.fc = nn.Sequential(
            nn.Linear(3,64),    # 输入状态维度3，隐藏层64个神经元
            nn.ReLU(),           # 激活函数ReLU
            nn.Linear(64,3),     # 输出3个动作
            nn.Softmax(dim=-1)   # softmax输出动作概率
        )
    def forward(self,x):
        return self.fc(x.float())  # 前向传播，保证输入为float32

# -------------------------------
# 初始化策略网络和优化器
# -------------------------------
policy_net = PolicyNet()                       # 策略网络
optimizer_pg = optim.Adam(policy_net.parameters(), lr=1e-2)  # Adam优化器

# 初始化记录列表
pg_rewards, pg_soc_history, pg_grid_history = [], [], []

# -------------------------------
# 策略梯度训练循环
# -------------------------------
for ep in range(total_episodes):
    s, _ = env_train.reset()                   # 重置训练环境，获取初始状态
    done = False                               # episode结束标志
    log_probs, rewards_list = [], []           # 存储动作的log概率和对应奖励
    total_reward = 0                           # 累计奖励
    env_train.soc_history, env_train.grid_history = [], []  # 清空历史记录

    while not done:
        # 计算当前状态下各动作的概率
        probs = policy_net(torch.tensor(s,dtype=torch.float32))
        # 根据概率构建分类分布
        dist = torch.distributions.Categorical(probs)
        # 从分布中采样动作
        a = dist.sample()
        # 与环境交互
        ns, r, terminated, truncated, _ = env_train.step(a.item())
        done = terminated or truncated
        # 保存动作的log概率，用于后续梯度更新
        log_probs.append(dist.log_prob(a))
        # 保存即时奖励
        rewards_list.append(r)
        s = ns                      # 状态更新
        total_reward += r            # 累加奖励

    # 计算折扣回报（G_t）
    G = 0
    returns = []
    for r in reversed(rewards_list):            # 从后向前计算累计折扣奖励
        G = r + gamma*G
        returns.insert(0,G)                     # 插入到列表开头
    returns = torch.tensor(returns, dtype=torch.float32)            # 转为tensor
    # 对returns进行标准化（减均值除标准差）提高训练稳定性
    returns = (returns - returns.mean()) / (returns.std() + 1e-9)

    # 计算策略梯度损失
    loss = 0
    for lp, R in zip(log_probs, returns):
        loss -= lp*R                            # REINFORCE公式：损失=-logπ(a|s)*G

    # 更新策略网络参数
    optimizer_pg.zero_grad()
    loss.backward()
    optimizer_pg.step()

    # 记录本轮奖励
    pg_rewards.append(total_reward)
    # 动态显示训练奖励曲线
    clear_output(wait=True)
    plot_reward_live(pg_rewards, title="Policy Gradient Live Reward")

# 绘制训练结束动画曲线
plot_reward_animation(pg_rewards,"Policy Gradient Training")

### 5.2 PG模型测试

In [ ]:
# -------------------------------
# Policy Gradient 测试阶段
# -------------------------------
s, _ = env_test.reset()        # 重置测试环境
done = False
pg_total_reward = 0
pg_test_rewards = []

while not done:
    with torch.no_grad():
    # 测试阶段仍根据策略网络输出动作概率采样动作
        probs = policy_net(torch.tensor(s,dtype=torch.float32))
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        ns, r, terminated, truncated, _ = env_test.step(a.item())
        done = terminated or truncated
        s = ns
        pg_total_reward += r             # 累加测试奖励
        pg_test_rewards.append(r)        # 保存奖励

pg_test_soc_history=env_test.soc_history.copy()
pg_test_grid_history=env_test.grid_history.copy()
pg_test_pv_history=env_test.pv_history.copy()
pg_test_load_history=env_test.load_history.copy()

print(f"Policy Gradient Test Total Reward: {pg_total_reward:.2f}")  # 输出测试总奖励

# 绘制测试阶段SOC、电网功率、光伏功率、负荷功率和奖励动画
plot_reward_animation(pg_test_rewards,"Policy Gradient Test")
plot_soc_power_animation(pg_test_soc_history, pg_test_grid_history, pg_test_load_history, pg_test_pv_history, title="Policy Gradient Test")


## 6 Actor-Critic(AC) 演员-评论家算法

### 6.1 AC算法构建与训练

In [ ]:
# -------------------------------
# Actor-Critic 网络定义
# -------------------------------
class ACNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 公共隐藏层：输入3维状态，输出64维隐藏特征
        self.fc = nn.Linear(3,64)
        # Actor头：输出3个动作概率
        self.actor = nn.Linear(64,3)
        # Critic头：输出状态价值（scalar）
        self.critic = nn.Linear(64,1)
    def forward(self,x):
        x = torch.relu(self.fc(x.float()))               # 全连接 + ReLU
        return torch.softmax(self.actor(x),-1), self.critic(x)  # 返回动作概率和状态价值

# -------------------------------
# 初始化AC网络和优化器
# -------------------------------
ac_net = ACNet()                                   # Actor-Critic网络
optimizer_ac = optim.Adam(ac_net.parameters(), lr=1e-2)  # Adam优化器

# 初始化记录列表
ac_rewards, ac_soc_history, ac_grid_history = [], [], []

# -------------------------------
# Actor-Critic训练循环
# -------------------------------
for ep in range(total_episodes):
    s, _ = env_train.reset()                       # 重置训练环境，获取初始状态
    done = False                                   # 回合结束标志
    total_reward = 0                               # 累计奖励
    env_train.soc_history, env_train.grid_history = [], []  # 清空历史记录

    while not done:
        # 前向传播：获取动作概率（Actor）和状态价值（Critic）
        probs, value = ac_net(torch.tensor(s,dtype=torch.float32))
        # 构建分类分布
        dist = torch.distributions.Categorical(probs)
        # 根据策略采样动作
        a = dist.sample()
        # 与环境交互
        ns, r, terminated, truncated, _ = env_train.step(a.item())
        done = terminated or truncated

        # 计算下一个状态的价值
        _, next_value = ac_net(torch.tensor(ns,dtype=torch.float32))
        # 计算TD目标（r + γ*V(s')）
        td_target = r + gamma * next_value * (1 - done)
        # 计算TD误差 δ = TD_target - V(s)
        td_error = td_target - value
        # Actor-Critic损失：
        # 1) Policy loss = -logπ(a|s) * δ.detach()
        # 2) Critic loss = δ^2
        loss = -dist.log_prob(a) * td_error.detach() + td_error.pow(2)

        # 更新网络参数
        optimizer_ac.zero_grad()
        loss.backward()
        optimizer_ac.step()

        s = ns                      # 状态更新
        total_reward += r            # 累加奖励

    # 每轮训练结束记录奖励和历史SOC/网购功率
    ac_rewards.append(total_reward)
    # 动态显示训练奖励曲线
    clear_output(wait=True)
    plot_reward_live(ac_rewards, title="Actor-Critic Live Reward")

# 绘制训练完成动画曲线
plot_reward_animation(ac_rewards,"Actor-Critic Training")


### 6.2 AC算法测试

In [ ]:
# -------------------------------
# Actor-Critic测试阶段
# -------------------------------
s, _ = env_test.reset()          # 重置测试环境
done = False
ac_total_reward = 0
ac_test_rewards=[]

while not done:
    with torch.no_grad():
    # 前向传播获取动作概率和状态价值
        probs, value = ac_net(torch.tensor(s,dtype=torch.float32))
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()             # 采样动作
        ns, r, terminated, truncated, _ = env_test.step(a.item())
        done = terminated or truncated
        s = ns
        ac_total_reward += r             # 累加测试奖励
        ac_test_rewards.append(r)        # 保存奖励

ac_test_soc_history=env_test.soc_history.copy()
ac_test_grid_history=env_test.grid_history.copy()
ac_test_pv_history=env_test.pv_history.copy()
ac_test_load_history=env_test.load_history.copy()

print(f"Actor-Critic Test Total Reward: {ac_total_reward:.2f}")  # 输出测试总奖励

# 绘制测试阶段SOC、电网功率、光伏功率、负荷功率和奖励动画
plot_reward_animation(ac_test_rewards,"Actor-Critic Test")
plot_soc_power_animation(ac_test_soc_history, ac_test_grid_history, ac_test_load_history, ac_test_pv_history, title="Actor-Critic Test")


## 7 四种策略的可视化对比

In [ ]:
test_rewards_all = {
    "Q-learning": q_total_reward,          # 假设你之前 Q-learning 测试结果保存到 total_reward
    "DQN": dqn_total_reward,            # DQN 测试结果
    "Policy Gradient": pg_total_reward, # PG 测试结果
    "Actor-Critic": ac_total_reward     # AC 测试结果
}

# -------------------------------
# 绘制奖励对比柱状图
plt.figure(figsize=(8,5))
plt.bar(test_rewards_all.keys(), test_rewards_all.values(), color=['blue','orange','green','red'])
plt.ylabel("Total Reward on Test Set")
plt.title("Four RL Strategies Test Reward Comparison")
plt.grid(axis='y')
plt.show()